In [ ]:
from pyspark.sql.functions import col, current_timestamp

In [ ]:
dbutils.widgets.text("catalog", "olist_project_dev")

dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("raw_olist_orders_payments_table", "olist_orders_payments")

dbutils.widgets.text("silver_schema", "olist_silver")
dbutils.widgets.text("orders_payments_table", "orders_payments_silver")

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_orders_payments_table_name = dbutils.widgets.get("raw_olist_orders_payments_table")

silver_schema = dbutils.widgets.get("silver_schema")
orders_payments_table_name = dbutils.widgets.get("orders_payments_table")

In [ ]:
raw_olist_orders_payments_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_orders_payments_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{orders_payments_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{orders_payments_table_name} (
            orderId STRING,
            paymentSequential INT,
            paymentType STRING,
            paymentInstallments INT,
            paymentValue DOUBLE,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
orders_payments_silver_df = (
    raw_olist_orders_payments_df
    .where((col("order_id").rlike("^[0-9a-fA-F]{32}$")))
    .select(
        col("order_id").cast("string").alias("orderId"),
        col("payment_sequential").cast("int").alias("paymentSequential"),
        col("payment_type").cast("string").alias("paymentType"),
        col("payment_installments").cast("int").alias("paymentInstallments"),
        col("payment_value").cast("double").alias("paymentValue")
    )
    .withColumn("processedTimestamp", current_timestamp())
)

In [ ]:
orders_payments_silver_df.createOrReplaceTempView("orders_payments_silver_view")

spark.sql(f"""
    MERGE INTO {catalog}.{silver_schema}.{orders_payments_table_name} AS target
    USING orders_payments_silver_view AS source
    ON target.orderId = source.orderId AND target.paymentSequential = source.paymentSequential
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
    """)